# Video downloading and frame extraction

This notebook shows how to:
- search and download walking-tour videos from YouTube using `yt-dlp`, and
- split each downloaded video into frames for later geolocalization processing.

## Use yt-dlp to collect candidate video URLs (command line)

### 1. Collect candidate YouTube URLs with yt-dlp (run in terminal)

Run the following command **in a terminal**. It performs YouTube searches and writes all matching video URLs into `./data/melbourne_URLs.txt`. Here we use Melbourne as an example.

```bash
queries=(
"Melbourne walk"
"Melbourne walking tour"
"Melbourne street walk"
"Melbourne city walk"
"Melbourne CBD walk"
)

(
for q in "${queries[@]}"
do
  yt-dlp "ytsearchall:$q" \
    --flat-playlist \
    --skip-download \
    --print "https://www.youtube.com/watch?v=%(id)s"

  sleep 2
done
) | sort -u | tee ./data/melbourne_URLs.txt
```

### 2. Download all videos by URLs with yt-dlp (run in terminal)

Before running the command:

1. Export cookies from your browser using a plugin such as **Get cookies.txt LOCALLY**.
2. Save the exported file as `cookies.txt` in the project root directory.

Next, run this command **in a terminal** to download every video listed in `./data/melbourne_URLs.txt`.
Videos will be stored under `./data/videos/`, filenames will use safe characters,
and a log of downloaded URLs will be appended to `./data/videos/downloaded_urls.txt`. The video parameter such as resolution can be adjusted accorrdingly. Please refer to the repo for more information: https://github.com/yt-dlp/yt-dlp or https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp

```bash
yt-dlp \
  --cookies ./cookies.txt \
  --user-agent "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36" \
  -a ./data/melbourne_URLs.txt \
  --download-archive ./data/archive.txt \
  --restrict-filenames \
  --write-description \
  --write-thumbnail \
  --sleep-interval 10 --max-sleep-interval 30 \
  --retries 50 --fragment-retries 50 \
  --retry-sleep "http:exp=2:180" \
  -f "bv*[height<=1080]+ba/b[height<=1080]/b" \
  -o "./videos/%(title)s/%(title)s.%(ext)s" \
  --print-to-file "%(webpage_url)s" "./data/downloaded_urls.txt"
```

# Video split to images frames for geolocalization

In [ ]:
# =================================================================
# NOTE: THIS IS A TERMINAL SCRIPT (NOT FOR JUPYTER CELL EXECUTION)
# =================================================================

import os
import subprocess
from pathlib import Path
from tqdm import tqdm

def main():
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
    
    VIDEO_ROOT = PROJECT_ROOT / "data" / "videos"
    IMAGE_ROOT = PROJECT_ROOT / "data" / "image"
    LOG_FILE = PROJECT_ROOT / "data" / "video_processing_log.txt"

    if not VIDEO_ROOT.exists():
        print(f"Error: {VIDEO_ROOT} not found.")
        return

    # Reset the previous log file
    LOG_FILE.write_text("") 

    # Discover all video series folders
    subfolders = [p for p in VIDEO_ROOT.iterdir() if p.is_dir()]

    if not subfolders:
        print(f"No video folders found in {VIDEO_ROOT}")
        return

    print(f"Starting video sampling: {len(subfolders)} series discovered.")

    with open(LOG_FILE, "a") as log:
        for folder in tqdm(subfolders, desc="Overall Progress"):
            series_name = folder.name
            
            output_folder = IMAGE_ROOT / series_name / "image"
            
            # Skip if already processed
            if output_folder.exists() and any(output_folder.iterdir()):
                log.write(f"skip {series_name}, output folder exists\n")
                continue

            # Find the single MP4 file
            mp4_files = list(folder.glob("*.mp4"))
            if len(mp4_files) != 1:
                log.write(f"skip {series_name}, no MP4 or multiple MP4s\n")
                continue

            video_path = mp4_files[0]
            output_folder.mkdir(parents=True, exist_ok=True)

            # FFmpeg Command Configuration
            output_pattern = output_folder / "frame_%04d.png"
            ffmpeg_command = [
                "ffmpeg", "-i", str(video_path),
                "-vf", "trim=start=90,setpts=PTS-STARTPTS,fps=1/10",
                "-q:v", "10",
                str(output_pattern)
            ]

            # Execute FFmpeg
            result = subprocess.run(ffmpeg_command, capture_output=True, text=True)

            if result.returncode == 0:
                log.write(f"succeed {series_name}\n")
            else:
                error_msg = f"failed: {series_name}\nreason: {result.stderr}\n"
                log.write(error_msg)
                print(f"\n[ERROR] {series_name} failed. Check log.")

    print(f"\nDone. Processing log saved to {LOG_FILE.relative_to(PROJECT_ROOT)}")

if __name__ == "__main__":
    main()